In [2]:
# BAYESIAN NETWORK FOR HEART DISEASE DIAGNOSIS

import pandas as pd

# 1. LOAD THE HEART DISEASE DATASET

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak",
    "slope", "ca", "thal", "num"
]

data = pd.read_csv(url, names=columns, na_values="?")

print("Dataset Loaded Successfully!")
print(data.head())

Dataset Loaded Successfully!
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  num  
0    3.0  0.0   6.0    0  
1    2.0  3.0   3.0    2  
2    2.0  2.0   7.0    1  
3    3.0  0.0   3.0    0  
4    1.0  0.0   3.0    0  


In [3]:
# 2. CLEAN THE DATA

data = data.apply(pd.to_numeric, errors="coerce")
data = data.dropna()

# Convert target into two classes
# 0 = No Heart Disease
# 1 = Heart Disease

data["HeartDisease"] = (data["num"] > 0).astype(int)

data = data.drop(columns=["num"])

print("\nCleaned Dataset:")
print(data.head())


Cleaned Dataset:
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  HeartDisease  
0    3.0  0.0   6.0             0  
1    2.0  3.0   3.0             1  
2    2.0  2.0   7.0             1  
3    3.0  0.0   3.0             0  
4    1.0  0.0   3.0             0  


In [4]:
# 3. DISCRETIZE CONTINUOUS FEATURES

data["age"] = pd.cut(
    data["age"],
    bins=[-float("inf"), 40, 60, float("inf")],
    labels=[0, 1, 2]
).astype(int)

data["trestbps"] = pd.cut(
    data["trestbps"],
    bins=[-float("inf"), 120, 140, float("inf")],
    labels=[0, 1, 2]
).astype(int)

data["chol"] = pd.cut(
    data["chol"],
    bins=[-float("inf"), 200, float("inf")],
    labels=[0, 1]
).astype(int)

data["thalach"] = pd.cut(
    data["thalach"],
    bins=[-float("inf"), 120, float("inf")],
    labels=[0, 1]
).astype(int)

data["oldpeak"] = pd.cut(
    data["oldpeak"],
    bins=[-float("inf"), 1, float("inf")],
    labels=[0, 1]
).astype(int)

In [5]:
# 4. CONSTRUCT THE BAYESIAN NETWORK

features = [
    "age", "sex", "cp", "trestbps", "chol",
    "fbs", "restecg", "thalach", "exang",
    "oldpeak", "slope", "ca", "thal"
]

target = "HeartDisease"

print("\nBayesian Network Structure:")

for feature in features:
    print(feature, "-->", target)


Bayesian Network Structure:
age --> HeartDisease
sex --> HeartDisease
cp --> HeartDisease
trestbps --> HeartDisease
chol --> HeartDisease
fbs --> HeartDisease
restecg --> HeartDisease
thalach --> HeartDisease
exang --> HeartDisease
oldpeak --> HeartDisease
slope --> HeartDisease
ca --> HeartDisease
thal --> HeartDisease


In [6]:
# 5. CALCULATE PRIOR PROBABILITIES

total = len(data)

prior = {}

for diagnosis in [0, 1]:

    count = len(data[data[target] == diagnosis])

    prior[diagnosis] = count / total

print("\nPrior Probabilities:")
print("No Heart Disease:", round(prior[0], 4))
print("Heart Disease:", round(prior[1], 4))


Prior Probabilities:
No Heart Disease: 0.5387
Heart Disease: 0.4613


In [7]:
# 6. CALCULATE CONDITIONAL PROBABILITIES

def calculate_probability(patient, diagnosis):

    probability = prior[diagnosis]

    # Get records belonging to the selected diagnosis
    class_data = data[data[target] == diagnosis]

    for feature in features:

        value = patient[feature]

        count = len(class_data[class_data[feature] == value])

        total_count = len(class_data)

        # Laplace smoothing
        states = data[feature].nunique()

        conditional_probability = (count + 1) / (total_count + states)

        probability = probability * conditional_probability

    return probability

In [8]:
# 7. DIAGNOSE A PATIENT

def diagnose_patient(patient):

    p_no = calculate_probability(patient, 0)
    p_yes = calculate_probability(patient, 1)

    # Normalize probabilities
    total_probability = p_no + p_yes

    p_no = p_no / total_probability
    p_yes = p_yes / total_probability

    if p_yes > p_no:
        result = "HEART DISEASE"
    else:
        result = "NO HEART DISEASE"

    print("\nProbability of No Heart Disease:", round(p_no, 4))
    print("Probability of Heart Disease:", round(p_yes, 4))
    print("Diagnosis:", result)

In [9]:
# 8. TEST WITH PATIENT 1

patient_1 = {
    "age": 2,
    "sex": 1,
    "cp": 4,
    "trestbps": 2,
    "chol": 1,
    "fbs": 0,
    "restecg": 2,
    "thalach": 0,
    "exang": 1,
    "oldpeak": 1,
    "slope": 2,
    "ca": 1,
    "thal": 3
}

print("\nPATIENT 1 DIAGNOSIS")
diagnose_patient(patient_1)


PATIENT 1 DIAGNOSIS

Probability of No Heart Disease: 0.001
Probability of Heart Disease: 0.999
Diagnosis: HEART DISEASE


In [10]:
# 9. TEST WITH PATIENT 2

patient_2 = {
    "age": 1,
    "sex": 0,
    "cp": 2,
    "trestbps": 1,
    "chol": 0,
    "fbs": 0,
    "restecg": 0,
    "thalach": 1,
    "exang": 0,
    "oldpeak": 0,
    "slope": 1,
    "ca": 0,
    "thal": 3
}

print("\nPATIENT 2 DIAGNOSIS")
diagnose_patient(patient_2)


print("\nBayesian Network Diagnosis Completed!")
print("Note: This is an educational demonstration, not a clinical diagnosis.")


PATIENT 2 DIAGNOSIS

Probability of No Heart Disease: 0.9992
Probability of Heart Disease: 0.0008
Diagnosis: NO HEART DISEASE

Bayesian Network Diagnosis Completed!
Note: This is an educational demonstration, not a clinical diagnosis.
